# Bước 12 (bản 2) – Đặt ngưỡng theo cửa sổ thời gian, thêm "Otsu chặn kép"

Ở bước 9 bản 3, cách c (khớp tỷ lệ) và d (Otsu) tính ngưỡng trên toàn bộ 900.000 luồng mới cùng lúc. Bước 12 làm giống thực tế hơn: xếp lưu lượng mới theo thời gian, cắt thành từng cửa sổ 20.000 luồng; ngưỡng cho mỗi cửa sổ chỉ được tính từ điểm số của 50.000 luồng ngay trước nó (không dùng nhãn).

| Chữ | Cách chọn ngưỡng |
|---|---|
| a | Ngưỡng chọn trên k nhãn, giữ nguyên suốt (cách cũ) |
| b | Khớp tỷ lệ trên phần thích nghi, giữ nguyên suốt |
| c_cua_so | Khớp tỷ lệ trên 50.000 luồng ngay trước |
| d_cua_so | Otsu trên 50.000 luồng ngay trước |
| e | Otsu có chặn: chỉ đổi ngưỡng khi điểm số quá khứ tách thành hai cụm rõ (độ tách ≥ 0,75); nếu không, giữ ngưỡng cũ |
| **f (mới)** | **Otsu chặn kép**: độ tách ≥ 0,6 VÀ thử ngưỡng mới trên các luồng bình thường có nhãn (mô hình chưa học chúng) – tỷ lệ báo nhầm không vượt 2% (hoặc mức báo nhầm của ngưỡng cố định + 1%) |
| g (mới) | Như f nhưng giữ độ tách ≥ 0,75 như e – để tách tác dụng của từng cái chặn |
| c_toan_bo, d_toan_bo | Như bước 9 bản 3 (dùng cả lô), để so sánh |

| Ô | Việc | Thời gian |
|---|---|---|
| 1 | Kiểm tra môi trường và mã nguồn | 1 phút |
| 2 | Chạy trên 4 mạng: mốc đủ nhãn + M2 với 500 / 1.000 / 5.000 nhãn, mỗi mức bốc 3 lần | 15–30 phút |

**Không cần GPU (card đồ họa).** Kết quả lưu vào `MyDrive/TADP_Q1/results/b12_cua_so_v2/` (thư mục mới). Nếu Colab ngắt, chạy lại Ô 2: mạng nào xong rồi sẽ được bỏ qua.

## Ô 1 – Kiểm tra môi trường và mã nguồn

In [1]:
import os, sys, subprocess
from google.colab import drive
drive.mount("/content/drive")
CODE_DIR = "/content/drive/MyDrive/TADP_Q1/code"
sys.path.insert(0, CODE_DIR)
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
try:
    import lightgbm, pyarrow
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lightgbm", "pyarrow"], check=True)
EXPECTED = {"__init__.py": 125, "cic_data.py": 8579, "config.py": 3263, "data.py": 9956, "experts.py": 5934,
            "few_label.py": 15932, "metrics.py": 4002, "preprocess.py": 3601, "shift.py": 3863,
            "window_threshold.py": 12330}
ok = True
for f, size in EXPECTED.items():
    p = os.path.join(CODE_DIR, "tadp", f)
    real = os.path.getsize(p) if os.path.exists(p) else None
    if real != size:
        ok = False
        print(f"   SAI: {f} – mong đợi {size} byte, thực tế {real}")
print("Mã nguồn:", "ĐẦY ĐỦ, đúng phiên bản bước 12 bản 2" if ok else "CÓ LỖI – báo lại cho Claude")

Mounted at /content/drive
Mã nguồn: ĐẦY ĐỦ, đúng phiên bản bước 12 bản 2


## Ô 2 – Chạy

In 4 dòng "[tên mạng] xong", rồi:
- Bảng 12.1: tỷ lệ tấn công thay đổi thế nào giữa các cửa sổ (bao nhiêu phần trăm cửa sổ gần như không có tấn công, bao nhiêu phần trăm gần như toàn tấn công);
- Bảng 12.2: Macro-F1 trung bình 4 mạng theo từng cách chọn ngưỡng;
- Bảng 12.3: Macro-F1 theo từng mạng;
- Bảng 12.4: tỷ lệ cửa sổ mà ngưỡng Otsu mới được nhận (cách e, f, g).

Chạy xong báo Claude.

In [2]:
import sys, warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
from google.colab import drive
drive.mount("/content/drive")
sys.path.insert(0, "/content/drive/MyDrive/TADP_Q1/code")
for m in [k for k in sys.modules if k.startswith("tadp")]:
    del sys.modules[m]
from tadp.config import get_config
from tadp.data import load_all, make_splits, common_features
from tadp.window_threshold import run_window
B = "/content/drive/MyDrive/TADP_Q1"
DOMAINS_CIC = {"CIC-IDS2017": ["2017"], "CSE-CIC-IDS2018": ["2018"], "CIC-ToN-IoT": ["ton"], "CIC-UNSW-NB15": ["unsw"]}
cfg = get_config(RAW_DIR=f"{B}/raw_cic", CACHE_DIR=f"{B}/cache_cic", RESULT_DIR=f"{B}/results/b12_cua_so_v2",
                 DOMAINS=DOMAINS_CIC, SPLIT_MODE="episode", LGBM_ROUNDS=200)
data = load_all(cfg)
splits, _ = make_splits(cfg, data)
cols = common_features(data, "no_id")
run_window(cfg, data, splits, cols, pv="P1_log")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CIC-IDS2017: 999,712 dòng, 87 cột, tỷ lệ tấn công = 0.2411
CSE-CIC-IDS2018: 999,829 dòng, 87 cột, tỷ lệ tấn công = 0.0559
CIC-ToN-IoT: 1,000,028 dòng, 82 cột, tỷ lệ tấn công = 0.5302
CIC-UNSW-NB15: 1,000,399 dòng, 82 cột, tỷ lệ tấn công = 0.0251
[CIC-IDS2017] xong trong 119 giây
[CSE-CIC-IDS2018] xong trong 121 giây
[CIC-ToN-IoT] xong trong 127 giây
[CIC-UNSW-NB15] xong trong 114 giây
BẢNG 12.1 – Tỷ lệ tấn công thay đổi thế nào giữa các cửa sổ 20,000 luồng của phần đánh giá
          target  so_cua_so  ty_le_tan_cong_tb  ty_le_tan_cong_thap_nhat  ty_le_tan_cong_cao_nhat  phan_tram_cua_so_duoi_1pc_tan_cong  phan_tram_cua_so_tren_99pc_tan_cong
    CIC-IDS2017         45              0.241                       0.0                    0.982                              51.111                                  0.0
CSE-CIC-IDS2018         45              0.056      

cach_nguong          a_co_dinh  b_khop_ty_le_thich_nghi  c_khop_ty_le_cua_so  c_toan_bo  d_otsu_cua_so  d_toan_bo  e_otsu_cua_so_co_chan  f_otsu_chan_kep  g_otsu_chan_kep_eta_cu
mo_hinh      k                                                                                                                                                                   
M2_hoc_moi   500         0.827                    0.834                0.753      0.904          0.869      0.915                  0.908            0.827                   0.829
             1000        0.784                    0.768                0.767      0.924          0.841      0.914                  0.882            0.791                   0.794
             5000        0.843                    0.881                0.752      0.943          0.865      0.942                  0.938            0.898                   0.899
tran_du_nhan 200000      0.970                    0.964                0.793      0.968          0.704      0.765                  0.765            0.970                   0.970